In [2]:
import pandas as pd
import numpy as np
from pathlib import Path


PROJECT_DIR = Path(
    r"C:\Users\adaly\OneDrive\Documents\DMDeficientGalaxyTNG-1"
)

CSV_FILE = PROJECT_DIR / "data" / "dmdg_catalog_z0.csv"


df = pd.read_csv(CSV_FILE)

print(f"Loaded {len(df):,} subhalos")

dmdgs = df[
    (df["DMDG_fDM_lt_0.5"]) &
    (df["M_star"] > 0)
].copy()

print(f"DMDG candidates: {len(dmdgs):,}")


h = 0.6774

dmdgs["M_star_Msun"] = (
    dmdgs["M_star"] * 1e10 / h
)

dmdgs["log_M_total_2Rh"] = np.log10(
    dmdgs["M_total_2Rh"]
)

dmdgs["log_M_star_Msun"] = np.log10(
    dmdgs["M_star_Msun"]
)

features = [
    "log_M_total_2Rh",
    "f_DM",
    "log_M_star_Msun"
]

dmdgs_non_extreme = dmdgs.dropna(
    subset=features
).copy()

dmdgs_non_extreme = dmdgs_non_extreme[
    np.isfinite(
        dmdgs_non_extreme[features]
    ).all(axis=1)
].copy()
dmdgs_non_extreme_final = dmdgs_non_extreme[
    dmdgs_non_extreme["f_DM"] > 0.05
]
print(
    f"Valid DMDGs for representative selection: "
    f"{len(dmdgs_non_extreme_final):,}"
)
z_score_cols = []

for col in features:

    median_val = dmdgs_non_extreme_final[col].median()

    mad_val = (
        dmdgs_non_extreme_final[col] - median_val
    ).abs().median()

    if mad_val == 0:
        mad_val = dmdgs_non_extreme_final[col].std()

    z_col = f"{col}_z"

    dmdgs_non_extreme_final[z_col] = (
        (dmdgs_non_extreme_final[col] - median_val)
        / mad_val
    ).abs()

    z_score_cols.append(z_col)
dmdgs_non_extreme_final["total_distance"] = np.sqrt(
    (
        dmdgs_non_extreme_final[z_score_cols] ** 2
    ).sum(axis=1)
)

dmdgs_z_sorted = dmdgs_non_extreme_final.sort_values(
    by="total_distance"
)

representative = dmdgs_z_sorted.head(5)
print()
print("=" * 70)
print("FIVE REPRESENTATIVE DMDGs")
print("=" * 70)

print(
    representative[
        [
            "SubhaloID",
            "GroupID",
            "M_star_Msun",
            "M_total_2Rh",
            "f_DM",
            "R_half_star",
            "total_distance"
        ]
    ].to_string(index=False)
)
output_file = PROJECT_DIR / "data" / "representative_dmdgs.csv"

representative.to_csv(
    output_file,
    index=False
)

print()
print(f"Saved to:")
print(output_file)

Loaded 13,922,457 subhalos
DMDG candidates: 41,918
Valid DMDGs for representative selection: 4,186

FIVE REPRESENTATIVE DMDGs
 SubhaloID  GroupID  M_star_Msun  M_total_2Rh     f_DM  R_half_star  total_distance
    738558     1159 2.589447e+09     0.272153 0.380555     0.684712        0.071609
    355308      229 2.492081e+09     0.244646 0.374496     1.563314        0.087032
    617680      750 2.661581e+09     0.254745 0.375286     0.773934        0.090881
   1436856    10494 2.075182e+09     0.306204 0.377263     0.666563        0.092623
    340851      211 2.726497e+09     0.270536 0.382829     1.231373        0.104302

Saved to:
C:\Users\adaly\OneDrive\Documents\DMDeficientGalaxyTNG-1\data\representative_dmdgs.csv


Imports and Generate Catalog

In [ ]:

features = ["M_total_2Rh", "f_DM", "M_star_Msun"]
z_score_cols = []
for col in features:
    median_val = dmdgs_non_extreme[col].median()
    
    mad_val = (dmdgs_non_extreme[col] - median_val).abs().median()
    
    if mad_val == 0: 
        mad_val = dmdgs_non_extreme[col].std()
    
    z_col = f"{col}_z"
    dmdgs_non_extreme[z_col] = ((dmdgs_non_extreme[col] - median_val) / mad_val).abs()
    z_score_cols.append(z_col)
dmdgs_non_extreme["total_distance"] = np.sqrt((dmdgs_non_extreme[z_score_cols] ** 2).sum(axis=1))
dmdgs_z_sorted = dmdgs_non_extreme.sort_values(by="total_distance")
sample_galaxy_one = dmdgs_z_sorted.iloc[0]
sample_galaxy_two = dmdgs_z_sorted.iloc[1]
sample_galaxy_three = dmdgs_z_sorted.iloc[2]
sample_galaxy_four = dmdgs_z_sorted.iloc[3]
sample_galaxy_five = dmdgs_z_sorted.iloc[4]

print(f"The most normal galaxy has a combined distance of: {sample_galaxy_one['total_distance']:.3f}")
print(f"{sample_galaxy_one}")
